<a href="https://colab.research.google.com/github/Mitul-Marimuthu/deep-learning/blob/project1/autograd_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data # actual number
    # how much this value affects the final loss (starts unknown)
    # filled in by backward
    self.grad = 0.0 # gradient starts at 0
    # the function that knows how to pass gradients to _prev
    self._backward = lambda: None # how to propogate gradient backward
    # builds the computation graph
    self._prev = set(_children) # what values created this one
    self._op = _op # for visualization (+, *, tanh, etc.)
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data:.4f}, grade={self.grad:.4f})"

  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      # gradient flows equally to both inputs
      self.grad += out.grad
      other.grad += out.grad
    out._backward = _backward

    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')

    def _backward():
      # chain rule: d(a*b)/da = b, d(a*b)/db = a
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad

    out._backward = _backward

    return out

  def __neg__(self):
    return self * -1

  def __sub__(self, other):
    return self + (-other)

  def __rmul__(self, other): # handles 2 * Value, not just Value * 2
    return self * other

  def __radd__(self, other):
    return self + other
